# Semiconductor Defect Prediction

## What we're building
A machine learning model that looks at process parameters (temperature, gas flow, etc.) during semiconductor manufacturing and **predicts whether a wafer will have a defect**.

This is a **binary classification** problem:
- Input: 10 process sensor readings
- Output: `0` (no defect) or `1` (defect)

## The ML workflow
Every ML project follows the same basic steps:
1. **Explore** the data — understand what you have
2. **Preprocess** — clean and transform it for a model
3. **Train** — fit a model to the training data
4. **Evaluate** — measure how well it works on unseen data
5. **Explain** — understand *why* the model makes its predictions

We'll go through each step here.

---
## Step 1: Load the Data

We use **pandas**, the standard Python library for tabular data. Think of it like Excel in code.

- `pd.read_csv()` reads a CSV into a **DataFrame** — a table with named columns
- `.head()` shows the first 5 rows
- `.shape` tells you (rows, columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('dataset/semiconductor_quality_control.csv')

print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

---
## Step 2: Explore the Data (EDA)

**EDA = Exploratory Data Analysis.** Before touching a model, always explore your data to:
- Check for missing values (nulls)
- Understand the data types
- See the distribution of your target variable
- Spot anything weird

### 2a. Data types and missing values

`.info()` is your first call on any new dataset — it shows column types and how many non-null values exist.

In [ ]:
df.info()

In [ ]:
# Check for missing values specifically
print('Missing values per column:')
print(df.isnull().sum())

### 2b. Summary statistics

`.describe()` gives you count, mean, min, max, and percentiles for every numeric column. This is how you quickly spot outliers or unexpected ranges.

In [ ]:
df.describe()

### 2c. Target variable distribution

The most important thing to check: **how balanced is your target?**

If 99% of rows are `0` and 1% are `1`, a model that always predicts `0` would look 99% accurate — but it's useless. We call this **class imbalance**.

In [ ]:
defect_counts = df['Defect'].value_counts()
defect_pct = df['Defect'].value_counts(normalize=True) * 100

print('Defect counts:')
print(defect_counts.rename({0: 'No Defect', 1: 'Defect'}))
print(f'\nDefect rate: {defect_pct[1]:.1f}%')

fig, ax = plt.subplots(figsize=(5, 4))
defect_counts.rename({0: 'No Defect', 1: 'Defect'}).plot(kind='bar', color=['steelblue', 'tomato'], ax=ax)
ax.set_title('Defect vs No Defect')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

### 2d. Defect rate by tool type

We have three tool types: Lithography, Deposition, Etching. Are some tools more prone to defects than others?

In [ ]:
tool_defect = df.groupby('Tool_Type')['Defect'].agg(['mean', 'count']).rename(columns={'mean': 'Defect Rate', 'count': 'Total Runs'})
tool_defect['Defect Rate'] = (tool_defect['Defect Rate'] * 100).round(1)
print(tool_defect)

tool_defect['Defect Rate'].plot(kind='bar', color='coral', figsize=(6, 4))
plt.title('Defect Rate by Tool Type')
plt.ylabel('Defect Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2e. Feature distributions — defect vs no defect

This is one of the most useful EDA plots: for each numeric feature, overlay the distribution for defective vs non-defective runs. If the distributions are separated, that feature is likely predictive.

In [ ]:
numeric_features = [
    'Chamber_Temperature', 'Gas_Flow_Rate', 'RF_Power', 'Etch_Depth',
    'Rotation_Speed', 'Vacuum_Pressure', 'Stage_Alignment_Error',
    'Vibration_Level', 'UV_Exposure_Intensity', 'Particle_Count'
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    df[df['Defect'] == 0][col].plot(kind='kde', ax=ax, label='No Defect', color='steelblue')
    df[df['Defect'] == 1][col].plot(kind='kde', ax=ax, label='Defect', color='tomato')
    ax.set_title(col.replace('_', ' '), fontsize=9)
    ax.legend(fontsize=7)
    ax.set_ylabel('')

plt.suptitle('Feature Distributions: Defect vs No Defect', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2f. Correlation heatmap

A **correlation matrix** shows how strongly pairs of variables move together.
- `1.0` = perfectly correlated (same direction)
- `-1.0` = perfectly anti-correlated (opposite direction)
- `0` = no linear relationship

We care most about the last row/column — which features correlate with `Defect`.

In [ ]:
corr_cols = numeric_features + ['Defect']
corr = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print('\nCorrelation with Defect (sorted):')
print(corr['Defect'].drop('Defect').sort_values(key=abs, ascending=False).round(3))

---
## Step 3: Preprocessing

Raw data isn't ready for a model. We need to:

1. **Drop non-informative columns** — IDs and timestamps don't help predict defects
2. **Encode categorical variables** — models need numbers, not text. We'll turn `Tool_Type` into numbers using **one-hot encoding** (a separate 0/1 column per category)
3. **Split into train/test sets** — we train on 80% of data and evaluate on the remaining 20% the model has never seen
4. **Scale features** — some models expect features to be on similar scales

### 3a. Select features and target

In [ ]:
# Drop columns that don't carry predictive signal
# Process_ID, Wafer_ID = just IDs; Timestamp = time; Join_Status = all the same value
drop_cols = ['Process_ID', 'Wafer_ID', 'Timestamp', 'Join_Status']
df_model = df.drop(columns=drop_cols)

print('Columns remaining:', list(df_model.columns))

### 3b. One-hot encode `Tool_Type`

**Why one-hot encoding?** If we encode Lithography=0, Deposition=1, Etching=2, the model would treat Etching as "greater than" Deposition, which is meaningless. One-hot encoding creates a separate binary column for each category — no false ordering.

In [ ]:
# pd.get_dummies creates a new column for each unique value in Tool_Type
# drop_first=True drops one column to avoid redundancy (if not Lithography and not Deposition, it must be Etching)
df_model = pd.get_dummies(df_model, columns=['Tool_Type'], drop_first=True)

print('Columns after encoding:', list(df_model.columns))
df_model.head(3)

### 3c. Train/test split

**Why split the data?** If we evaluate our model on the same data we trained it on, it could just memorize the answers — 100% accuracy, completely useless in the real world. The test set simulates data the model has never seen.

- `test_size=0.2` → 20% held out for testing
- `stratify=y` → ensures the defect rate is the same in both splits (important for imbalanced data)
- `random_state=42` → fixes the randomness so results are reproducible

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['Defect'])  # features (inputs)
y = df_model['Defect']                 # target (what we're predicting)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} rows ({y_train.mean()*100:.1f}% defect)')
print(f'Test set:      {X_test.shape[0]} rows ({y_test.mean()*100:.1f}% defect)')
print(f'Features:      {X_train.shape[1]}')

### 3d. Feature scaling

**Why scale?** Look at the ranges: `Chamber_Temperature` is around 70–80, but `Rotation_Speed` is around 1300–1500. Some models (like Logistic Regression) are sensitive to these scale differences — features with large values dominate the math.

**StandardScaler** transforms each feature to have mean=0 and standard deviation=1.

> **Important rule:** Fit the scaler *only on training data*, then apply it to test data. If you include test data in the fit, you're "leaking" information about the test set into your model — a common mistake.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit AND transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test (no fit!)

print('Scaling done.')
print(f'Before scaling — Chamber_Temperature mean: {X_train["Chamber_Temperature"].mean():.2f}')
print(f'After scaling  — Column 0 mean: {X_train_scaled[:, 0].mean():.4f} (≈ 0)')

---
## Step 4: Train a Baseline Model — Logistic Regression

Before training a powerful model, always start with a **simple baseline**. Here we use **Logistic Regression** — despite its name, it's a classification model.

**How it works (conceptually):**
It learns a weighted sum of the features, then squashes that sum through a sigmoid function to get a probability between 0 and 1.
- `probability > 0.5` → predict defect
- `probability ≤ 0.5` → predict no defect

The baseline tells us: what's the minimum bar a more complex model needs to beat?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]  # probability of class 1 (defect)

print('=== Logistic Regression ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lr):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.3f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['No Defect', 'Defect']))

### Understanding the metrics

- **Accuracy** — % of predictions that are correct. Can be misleading with imbalanced data.
- **Precision** (for Defect class) — of all the runs the model *flagged* as defective, what % actually were? High precision = few false alarms.
- **Recall** (for Defect class) — of all the runs that *were* defective, what % did the model catch? High recall = few missed defects.
- **F1 score** — the harmonic mean of precision and recall. Good single-number summary.
- **ROC-AUC** — measures how well the model ranks defective runs above non-defective ones. 1.0 = perfect, 0.5 = random guessing.

In manufacturing, **recall matters more** — missing a real defect (false negative) is worse than a false alarm (false positive).

In [ ]:
# Confusion matrix — shows exactly what the model got right and wrong
# Rows = actual, Columns = predicted
# Top-left = True Negatives (correctly said no defect)
# Bottom-right = True Positives (correctly caught defects)
# Bottom-left = False Negatives (missed defects — dangerous!)
# Top-right = False Positives (false alarms)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=['No Defect', 'Defect'],
    cmap='Blues', ax=ax
)
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## Step 5: Train a Better Model — XGBoost

**XGBoost** (Extreme Gradient Boosting) is one of the most powerful algorithms for tabular data. It's the go-to for Kaggle competitions and industry ML.

**How it works (conceptually):**
It builds many small decision trees **sequentially**, where each new tree focuses on correcting the mistakes of all the previous ones. The final prediction is the combined vote of all trees.

Unlike Logistic Regression, XGBoost:
- Handles non-linear relationships automatically
- Is robust to unscaled features (we'll use the unscaled data)
- Handles class imbalance via `scale_pos_weight`

**`scale_pos_weight`**: Since only 14.6% of runs are defects, we tell XGBoost to weight defect examples more heavily so it doesn't just predict "no defect" for everything.

In [ ]:
from xgboost import XGBClassifier

# Calculate the weight: ratio of negative to positive examples
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_weight = neg / pos
print(f'Negative examples: {neg}, Positive (defect): {pos}')
print(f'scale_pos_weight: {scale_weight:.2f}')

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,  # handle class imbalance
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

# XGBoost doesn't need scaled data
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print('\n=== XGBoost ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_xgb):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_xgb):.3f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['No Defect', 'Defect']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb, display_labels=['No Defect', 'Defect'],
    cmap='Greens', ax=ax
)
ax.set_title('XGBoost — Confusion Matrix')
plt.tight_layout()
plt.show()

### ROC Curve

The **ROC curve** plots **True Positive Rate** (recall) vs **False Positive Rate** as you vary the decision threshold. A model that's better at separating classes has a curve that hugs the top-left corner.

- **AUC = Area Under the Curve** — the higher the better. AUC=1.0 is perfect.
- The dashed diagonal line is a random classifier (AUC=0.5).

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 5))

RocCurveDisplay.from_predictions(y_test, y_prob_lr, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_xgb, name='XGBoost', ax=ax)

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## Step 6: Feature Importance

XGBoost internally tracks which features it used the most when making splits in its trees. This gives us **feature importance** — a score for each feature indicating how much it contributed to the model's predictions.

This is useful for:
- Understanding the model
- Identifying which process parameters most influence defects
- Potentially dropping unimportant features to simplify the model

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('XGBoost Feature Importance')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Feature importance (most to least):')
print(importances.sort_values(ascending=False).round(4))

---
## Step 7: SHAP — Explain Individual Predictions

Feature importance tells you which features matter *on average across all predictions*. But what about a specific prediction? Why did the model flag *this particular run* as defective?

**SHAP (SHapley Additive exPlanations)** answers exactly that. It assigns each feature a contribution value for each individual prediction:
- **Positive SHAP value** → pushed the prediction toward *defect*
- **Negative SHAP value** → pushed the prediction toward *no defect*

The starting point for every prediction is the base rate (average prediction), and SHAP values explain how far each feature pushed the result away from that base.

### 7a. SHAP summary plot

Shows all features across all test samples. Each dot is one prediction. Color = feature value (red = high, blue = low). Horizontal position = SHAP value (right = pushed toward defect).

In [ ]:
import shap
shap.initjs()

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, plot_type='dot', show=False)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plt.show()

### 7b. SHAP bar plot (mean absolute impact)

A simpler view: the average absolute SHAP value for each feature — the overall impact on the model output.

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Feature Impact (Mean |SHAP value|)')
plt.tight_layout()
plt.show()

### 7c. Explain a single prediction

A **waterfall plot** shows exactly why the model made a specific prediction for one sample. 
- Start at the base value (average model output)
- Each bar shows how much one feature pushed the prediction up or down
- The final value is the model's output for this sample

In [ ]:
# Find the index of the first actual defect in the test set
defect_idx = y_test[y_test == 1].index[0]
test_position = list(X_test.index).index(defect_idx)

predicted = xgb.predict(X_test.iloc[[test_position]])[0]
probability = xgb.predict_proba(X_test.iloc[[test_position]])[0][1]
actual = y_test.iloc[test_position]

print(f'Actual:    {"Defect" if actual == 1 else "No Defect"}')
print(f'Predicted: {"Defect" if predicted == 1 else "No Defect"} (probability: {probability:.1%})')

shap_explanation = shap.Explanation(
    values=shap_values[test_position],
    base_values=explainer.expected_value,
    data=X_test.iloc[test_position],
    feature_names=X_test.columns.tolist()
)

plt.figure()
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f'Why did the model predict Defect? (actual: {"Defect" if actual else "No Defect"})')
plt.tight_layout()
plt.show()

---
## Step 8: Summary & What to Do Next

Let's print a final comparison of both models.

In [ ]:
from sklearn.metrics import f1_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    'F1 (Defect)': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_xgb)
    ],
    'Recall (Defect)': [
        __import__('sklearn.metrics', fromlist=['recall_score']).recall_score(y_test, y_pred_lr),
        __import__('sklearn.metrics', fromlist=['recall_score']).recall_score(y_test, y_pred_xgb)
    ]
}).set_index('Model').round(3)

print('=== Final Model Comparison ===')
print(results.to_string())

## What you learned

| Concept | What it is |
|---|---|
| Binary classification | Predicting one of two outcomes (defect/no defect) |
| EDA | Exploring data before modeling to understand distributions and relationships |
| One-hot encoding | Converting categorical text columns to numeric 0/1 columns |
| Train/test split | Holding out data the model never sees to evaluate real-world performance |
| StandardScaler | Normalizing feature scales so no one feature dominates |
| Logistic Regression | Simple linear baseline classifier |
| XGBoost | Gradient boosted trees — powerful default for tabular data |
| Precision / Recall | Tradeoff between false alarms vs missed defects |
| ROC-AUC | Overall ranking ability of the model (threshold-independent) |
| Feature importance | Which inputs the model relied on most |
| SHAP | Why the model made a *specific* prediction |

## Next steps
- **Tune hyperparameters** with `GridSearchCV` or `RandomizedSearchCV` to improve XGBoost further
- **Try Random Forest** (`sklearn.ensemble.RandomForestClassifier`) as another comparison
- **Adjust the threshold** — instead of predicting defect at 50% probability, try 30% to catch more defects at the cost of more false alarms
- **Build a prediction UI** — wrap the model in a Streamlit app where you input process parameters and get a live prediction

---
---
# Part 2: Making the Model Better

The model we built works, but we made several decisions by guessing — the number of trees, the learning rate, the max depth. And we only evaluated it on one train/test split, which might have gotten lucky (or unlucky).

This section covers three skills every ML engineer needs:

1. **Cross-validation** — getting a reliable, unbiased estimate of model performance
2. **Hyperparameter tuning** — systematically searching for the best model settings
3. **Threshold optimization** — choosing the right decision cutoff for your specific problem

---
## Step 9: Cross-Validation

### The problem with a single train/test split

When we did `train_test_split`, we randomly put 20% of data aside as the test set. But what if that 20% happened to be unusually easy — or unusually hard? Our performance estimate would be misleading.

**Cross-validation (CV)** solves this by splitting the data multiple times and averaging the results.

**How k-fold CV works:**
1. Divide training data into k equal chunks ("folds") — usually k=5 or k=10
2. Train on k-1 folds, evaluate on the remaining 1 fold
3. Repeat k times, each time using a different fold as the holdout
4. Average the k scores

You get a performance estimate that's much less sensitive to any one lucky/unlucky split, plus a standard deviation so you know how stable the model is.

> The test set we created earlier is still kept completely separate — CV only runs on the training data. The test set is used only for the final evaluation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# StratifiedKFold preserves the defect rate in every fold — important for imbalanced data
# (regular KFold doesn't guarantee this)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_validate runs the full train/evaluate loop k times
# scoring='roc_auc' uses AUC as the metric; you can pass a list for multiple metrics
cv_results = cross_validate(
    xgb, X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1', 'recall'],
    return_train_score=True  # also score on training folds to check for overfitting
)

print('=== 5-Fold Cross-Validation Results (on training data) ===\n')
for metric in ['roc_auc', 'f1', 'recall']:
    test_scores  = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    print(f'{metric.upper():10s}  '
          f'CV mean: {test_scores.mean():.3f} ± {test_scores.std():.3f}  |  '
          f'Train mean: {train_scores.mean():.3f}')

print('\nIndividual fold AUC scores:', cv_results['test_roc_auc'].round(3))

### Reading the CV results

- **CV mean ± std**: The average score across 5 folds ± standard deviation. A small std means the model performs consistently regardless of which data it trains on — that's a good sign.
- **Train mean vs CV mean**: If train score is much higher than CV score, the model is **overfitting** — memorizing training data rather than learning general patterns. If they're close, the model generalizes well.

**Overfitting** is one of the most important concepts in ML. A model that overfits looks great on training data but fails on new data. Cross-validation is how you detect it.

---
## Step 10: Hyperparameter Tuning with RandomizedSearchCV

### What are hyperparameters?

A model has two kinds of parameters:
- **Parameters** — values the model *learns* from data (e.g. the weights in logistic regression)
- **Hyperparameters** — settings you choose *before* training that control how the model learns (e.g. how many trees, how deep each tree can grow, the learning rate)

We guessed these in Part 1. Now we'll search for better values systematically.

### Grid Search vs Randomized Search

- **GridSearchCV** — tries every possible combination of hyperparameters. Thorough but slow. With 4 params and 5 values each, that's 5⁴ = 625 combinations × k folds.
- **RandomizedSearchCV** — randomly samples combinations from the search space. Much faster, and research shows it often finds just as good results because most hyperparameter space is flat.

**Key hyperparameters for XGBoost:**
| Hyperparameter | What it controls |
|---|---|
| `n_estimators` | Number of trees. More = potentially better but slower and can overfit |
| `max_depth` | How deep each tree can grow. Deeper = more complex, higher overfit risk |
| `learning_rate` | How much each new tree corrects the previous ones. Lower = more trees needed but often more accurate |
| `subsample` | Fraction of training rows used per tree. < 1.0 adds randomness, reduces overfitting |
| `colsample_bytree` | Fraction of features used per tree. < 1.0 adds randomness, reduces overfitting |
| `min_child_weight` | Minimum data needed in a leaf. Higher = simpler trees, less overfit |

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define the search space — ranges/lists of values to sample from
# scipy's randint(a, b) samples integers from [a, b)
# scipy's uniform(a, b) samples floats from [a, a+b]
param_dist = {
    'n_estimators':     randint(100, 600),
    'max_depth':        randint(3, 9),
    'learning_rate':    uniform(0.01, 0.3),    # samples from [0.01, 0.31]
    'subsample':        uniform(0.6, 0.4),     # samples from [0.6, 1.0]
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),
}

base_xgb = XGBClassifier(
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

search = RandomizedSearchCV(
    base_xgb,
    param_distributions=param_dist,
    n_iter=50,              # try 50 random combinations
    scoring='roc_auc',      # optimize for AUC
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,              # use all CPU cores in parallel
    verbose=1
)

print('Running randomized search (50 combinations × 5 folds = 250 fits)...')
search.fit(X_train, y_train)

print(f'\nBest AUC (CV): {search.best_score_:.4f}')
print('\nBest hyperparameters found:')
for param, value in search.best_params_.items():
    print(f'  {param}: {value}')

In [ ]:
# Evaluate the tuned model on the held-out test set
best_xgb = search.best_estimator_

y_pred_tuned = best_xgb.predict(X_test)
y_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]

print('=== Tuned XGBoost vs Original XGBoost ===\n')
print(f"{'Metric':<20} {'Original':>10} {'Tuned':>10}")
print('-' * 42)
print(f"{'Accuracy':<20} {accuracy_score(y_test, y_pred_xgb):>10.3f} {accuracy_score(y_test, y_pred_tuned):>10.3f}")
print(f"{'ROC-AUC':<20} {roc_auc_score(y_test, y_prob_xgb):>10.3f} {roc_auc_score(y_test, y_prob_tuned):>10.3f}")
print(f"{'F1 (Defect)':<20} {f1_score(y_test, y_pred_xgb):>10.3f} {f1_score(y_test, y_pred_tuned):>10.3f}")

from sklearn.metrics import recall_score
print(f"{'Recall (Defect)':<20} {recall_score(y_test, y_pred_xgb):>10.3f} {recall_score(y_test, y_pred_tuned):>10.3f}")

### What to take away from this

- **Even small AUC improvements matter** — going from 0.92 to 0.94 sounds small but means the model is ranking defective wafers much better relative to good ones.
- **Tuning doesn't always give a huge boost** — if the original model was already well-configured, the gains will be modest. That's fine; the skill is in knowing *how* to tune, not just chasing big numbers.
- **`n_jobs=-1`** — always pass this when doing search. It parallelizes across all CPU cores and can cut runtime by 4-8x.

---
## Step 11: Threshold Optimization

### The default threshold is almost always wrong

Every classifier outputs a **probability** — e.g. "72% chance of defect." To turn that into a yes/no prediction, you apply a **threshold**: predict defect if probability > threshold.

The default is `0.5`. But 0.5 is just a convention — it's rarely the right choice.

**The tradeoff:**
- **Lower threshold** (e.g. 0.3): flag more runs as defective → catch more real defects (higher recall) → but also more false alarms (lower precision). Good when missing a defect is expensive.
- **Higher threshold** (e.g. 0.7): only flag when very confident → fewer false alarms (higher precision) → but miss more real defects (lower recall).

In semiconductor manufacturing, **a missed defect is far more costly** than a false alarm — a defective chip that ships causes warranty failures, recalls, reputation damage. So we want high recall, even at the cost of some precision.

### The Precision-Recall curve

Instead of picking one threshold blindly, we plot precision and recall for *every possible threshold* and find the best one for our needs.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob_tuned)
avg_precision = average_precision_score(y_test, y_prob_tuned)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Precision-Recall curve
ax = axes[0]
ax.plot(recall_vals, precision_vals, color='purple', lw=2)
ax.axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline (always predict defect): {y_test.mean():.2f}')
ax.set_xlabel('Recall (fraction of real defects caught)')
ax.set_ylabel('Precision (fraction of flagged runs that are real defects)')
ax.set_title(f'Precision-Recall Curve (AP = {avg_precision:.3f})')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

# Right: Precision and recall vs threshold — lets you pick a threshold visually
ax2 = axes[1]
ax2.plot(thresholds, precision_vals[:-1], label='Precision', color='steelblue', lw=2)
ax2.plot(thresholds, recall_vals[:-1], label='Recall', color='tomato', lw=2)
ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.7, label='Default threshold (0.5)')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Score')
ax2.set_title('Precision & Recall vs Threshold')
ax2.legend()
ax2.set_xlim([0, 1])

plt.tight_layout()
plt.show()

### Finding the optimal threshold programmatically

We want to catch as many real defects as possible (high recall) while keeping precision acceptable (we don't want to shut down the line for every single run).

A common approach: find the threshold that **maximizes F1** — the harmonic mean of precision and recall. This is a reasonable default if you don't have a specific business requirement.

But you can also set a **minimum recall** you require (e.g. "we must catch at least 90% of defects") and find the highest-precision threshold that satisfies that constraint.

In [ ]:
import numpy as np

# --- Method 1: threshold that maximizes F1 ---
f1_scores = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (precision_vals[:-1] + recall_vals[:-1] + 1e-9)
best_f1_idx = f1_scores.argmax()
best_f1_threshold = thresholds[best_f1_idx]

# --- Method 2: highest-precision threshold with recall >= 0.90 ---
min_recall = 0.90
valid = recall_vals[:-1] >= min_recall
if valid.any():
    best_recall_constrained_idx = precision_vals[:-1][valid].argmax()
    constrained_threshold = thresholds[valid][best_recall_constrained_idx]
else:
    constrained_threshold = None

print('=== Threshold Analysis ===\n')
print(f'Default threshold (0.5):')
y_pred_default = (y_prob_tuned >= 0.5).astype(int)
print(f'  Precision: {precision_vals[:-1][np.abs(thresholds - 0.5).argmin()]:.3f}')
print(f'  Recall:    {recall_vals[:-1][np.abs(thresholds - 0.5).argmin()]:.3f}')
print(f'  F1:        {f1_scores[np.abs(thresholds - 0.5).argmin()]:.3f}')

print(f'\nBest F1 threshold ({best_f1_threshold:.3f}):')
y_pred_f1 = (y_prob_tuned >= best_f1_threshold).astype(int)
print(f'  Precision: {precision_vals[best_f1_idx]:.3f}')
print(f'  Recall:    {recall_vals[best_f1_idx]:.3f}')
print(f'  F1:        {f1_scores[best_f1_idx]:.3f}')

if constrained_threshold is not None:
    print(f'\nMax-precision threshold with recall ≥ {min_recall} ({constrained_threshold:.3f}):')
    y_pred_constrained = (y_prob_tuned >= constrained_threshold).astype(int)
    p = precision_vals[:-1][valid][best_recall_constrained_idx]
    r = recall_vals[:-1][valid][best_recall_constrained_idx]
    print(f'  Precision: {p:.3f}')
    print(f'  Recall:    {r:.3f}')
    print(f'  F1:        {2*p*r/(p+r):.3f}')

In [ ]:
# Visualize what the optimized threshold actually looks like on the confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

y_pred_opt = (y_prob_tuned >= best_f1_threshold).astype(int)

ConfusionMatrixDisplay.from_predictions(
    y_test, (y_prob_tuned >= 0.5).astype(int),
    display_labels=['No Defect', 'Defect'], cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Default threshold (0.5)')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opt,
    display_labels=['No Defect', 'Defect'], cmap='Purples', ax=axes[1]
)
axes[1].set_title(f'Optimized threshold ({best_f1_threshold:.2f})')

plt.suptitle('Effect of Threshold Optimization on Confusion Matrix', y=1.02)
plt.tight_layout()
plt.show()

print('Look at the bottom-left cell (False Negatives = missed defects).')
print('That number should go down with the optimized threshold.')

---
## Final Summary: What improved?

Let's do one clean final comparison across all three model versions.

| Version | What changed |
|---|---|
| **Logistic Regression** | Simple baseline — linear model |
| **XGBoost (default)** | Gradient boosting, guessed hyperparameters |
| **XGBoost (tuned)** | Same algorithm, hyperparameters found via RandomizedSearchCV |
| **XGBoost (tuned + threshold)** | Same model, decision threshold optimized for manufacturing context |

In [ ]:
from sklearn.metrics import precision_score

models = {
    'Logistic Regression':        (y_pred_lr,    y_prob_lr),
    'XGBoost (default)':          (y_pred_xgb,   y_prob_xgb),
    'XGBoost (tuned)':            (y_pred_tuned, y_prob_tuned),
    'XGBoost (tuned + threshold)': (y_pred_opt,  y_prob_tuned),
}

rows = []
for name, (y_pred, y_prob) in models.items():
    rows.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
    })

final = pd.DataFrame(rows).set_index('Model').round(3)
print('=== Final Model Comparison ===')
print(final.to_string())
print('\nKey insight: Recall = fraction of real defects caught.')
print('In manufacturing, this is the number that matters most.')